# 🎙️ Clonador de Voz con F5-TTS
### Generador de audio para videos de terror — **Totalmente GRATIS con Google Colab**

---

## 📋 Instrucciones ANTES de empezar:

1. **Activa la GPU:** Ve a `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → selecciona **GPU T4** → Guardar
2. **Ejecuta las celdas en orden** haciendo clic en el botón ▶️ de cada celda
3. **Ten listo:** Un audio de tu voz de al menos 10-30 segundos (limpio, sin música de fondo)

---
⚠️ **IMPORTANTE:** El audio de referencia debe ser claro, sin ruido de fondo ni música. Si lo grabas específicamente para esto, mejor.

In [ ]:
# ============================================================
# CELDA 1: Verificar GPU y preparar el entorno
# ▶️ Ejecuta esta celda primero. Tarda ~2 minutos.
# ============================================================

import subprocess
import sys

# Verificar GPU
print('🔍 Verificando GPU...')
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                        capture_output=True, text=True)
if result.returncode == 0:
    print(f'✅ GPU detectada: {result.stdout.strip()}')
else:
    print('❌ NO se detectó GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU T4')
    print('   Sin GPU el proceso tardará 30-60 minutos por minuto de audio.')

print()
print('📦 Instalando F5-TTS y dependencias... (esto tarda ~3-5 minutos la primera vez)')
print('   Por favor espera sin cerrar la pestaña...')
print()

# Instalar F5-TTS
subprocess.run([sys.executable, '-m', 'pip', 'install', 'f5-tts', '-q'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'soundfile', 'pydub', '-q'], check=True)

print()
print('✅ ¡Todo instalado correctamente!')
print('   Continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 2: Subir tu audio de referencia (TU VOZ)
# ▶️ Ejecuta esta celda y sube tu archivo de audio
# ============================================================

from google.colab import files
import shutil
import os

print('📁 Se abrirá un selector de archivos...')
print('   Sube un archivo de audio con tu voz (.mp3 o .wav)')
print('   Requisitos: mínimo 10 segundos, sin música de fondo\n')

uploaded = files.upload()

if not uploaded:
    print('❌ No subiste ningún archivo. Vuelve a ejecutar esta celda.')
else:
    ref_audio_original = list(uploaded.keys())[0]
    ref_audio_path = '/content/mi_voz_referencia.wav'
    
    # Convertir a WAV si es MP3
    if ref_audio_original.lower().endswith('.mp3'):
        print('🔄 Convirtiendo MP3 a WAV...')
        from pydub import AudioSegment
        audio = AudioSegment.from_mp3(ref_audio_original)
        audio = audio.set_channels(1).set_frame_rate(24000)
        audio.export(ref_audio_path, format='wav')
        print('✅ Convertido a WAV correctamente.')
    else:
        shutil.copy(ref_audio_original, ref_audio_path)
        print(f'✅ Audio de referencia guardado como: {ref_audio_path}')
    
    print()
    print(f'🎙️ Archivo de referencia listo: {ref_audio_path}')
    print('   Continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 3: Escribe qué dice tu audio de referencia
# ▶️ Ejecuta esta celda y escribe la transcripción exacta
# ============================================================

# ⚠️ IMPORTANTE: Escribe aquí EXACTAMENTE lo que dices en el audio que subiste
# Cuanto más precisa sea la transcripción, mejor será la calidad de la voz clonada

TEXTO_DEL_AUDIO_REFERENCIA = """Escribe aquí exactamente lo que dices en tu audio de referencia.
Por ejemplo: Hola, soy Nelson y este es mi canal de terror donde exploramos los misterios más oscuros del mundo."""

# -------------------------------------------------------

print('📝 Texto de referencia configurado:')
print(f'   "{TEXTO_DEL_AUDIO_REFERENCIA[:100]}..."' if len(TEXTO_DEL_AUDIO_REFERENCIA) > 100 else f'   "{TEXTO_DEL_AUDIO_REFERENCIA}"')
print()
print('✅ Si el texto es correcto, continúa con la siguiente celda ▼')
print('   Si necesitas cambiarlo, edita la variable TEXTO_DEL_AUDIO_REFERENCIA arriba y vuelve a ejecutar.')

In [ ]:
# ============================================================
# CELDA 4: ✍️ ESCRIBE TU GUIÓN AQUÍ
# ▶️ Pega el guion optimizado de tu app y ejecuta
# ============================================================

# 💡 CONSEJO: Pega aquí el guion optimizado que te generó tu app de análisis
# Puedes dividirlo en párrafos para generar secciones separadas

GUION_A_NARRAR = """Bienvenidos a otro video donde la realidad supera a la ficción.
Esta es la historia de una casa abandonada en las afueras de la ciudad, donde los vecinos
aseguran haber escuchado voces en la oscuridad. Nadie sabe qué ocurrió realmente aquella noche.
Pero lo que está a punto de descubrir cambiará su perspectiva para siempre."""

# -------------------------------------------------------

print('📖 Guion a narrar:')
print('-' * 60)
print(GUION_A_NARRAR)
print('-' * 60)
print(f'   Total de caracteres: {len(GUION_A_NARRAR)}')
palabras = len(GUION_A_NARRAR.split())
print(f'   Palabras aproximadas: {palabras}')
print(f'   Duración estimada del audio: ~{palabras // 130} min {(palabras % 130) * 60 // 130} seg')
print()
print('✅ Si el guion es correcto, continúa con la siguiente celda ▼')

In [ ]:
# ============================================================
# CELDA 5: 🚀 GENERAR AUDIO CON TU VOZ
# ▶️ Ejecuta esta celda para generar el audio
# ⏱️ Tarda ~1-3 minutos dependiendo del largo del guion
# ============================================================

import time
import soundfile as sf
import numpy as np

OUTPUT_PATH = '/content/audio_generado.wav'

print('🎙️ Iniciando clonación de voz con F5-TTS...')
print('   Cargando modelo (solo la primera vez tarda ~2 min)...')
print()

try:
    from f5_tts.api import F5TTS
    
    # Inicializar el modelo
    f5tts = F5TTS()
    
    print('✅ Modelo cargado.')
    print('🔄 Generando audio con tu voz...')
    start_time = time.time()
    
    # Generar el audio
    wav, sr, spect = f5tts.infer(
        ref_file=ref_audio_path,
        ref_text=TEXTO_DEL_AUDIO_REFERENCIA,
        gen_text=GUION_A_NARRAR,
        file_wave=OUTPUT_PATH,
        seed=42  # Para reproducibilidad
    )
    
    elapsed = time.time() - start_time
    print(f'✅ ¡Audio generado en {elapsed:.1f} segundos!')
    print(f'   Guardado en: {OUTPUT_PATH}')
    
except Exception as e:
    print(f'❌ Error: {e}')
    print()
    print('💡 Si el error dice "ref_audio_path no definido", ejecuta primero la Celda 2.')
    print('   Si el error es de memoria, el guion es muy largo. Divídelo en partes más pequeñas.')

In [ ]:
# ============================================================
# CELDA 6: 🔊 ESCUCHAR Y DESCARGAR EL AUDIO
# ▶️ Ejecuta para reproducir y descargar tu audio
# ============================================================

import IPython.display as ipd
from google.colab import files
import os

if os.path.exists(OUTPUT_PATH):
    print('🔊 Reproduciendo audio generado...')
    print()
    display(ipd.Audio(OUTPUT_PATH))
    print()
    print('📥 Descargando archivo...')
    files.download(OUTPUT_PATH)
    print('✅ ¡Listo! El archivo se descargó como "audio_generado.wav"')
    print()
    print('💡 TIP: Importa este archivo en tu editor de video (CapCut, DaVinci, Premiere)')
    print('   y acomódalo con las imágenes de tu video de terror.')
else:
    print('❌ No se encontró el archivo de audio. Ejecuta primero la Celda 5.')

In [ ]:
# ============================================================
# CELDA EXTRA (Opcional): Generar en PARTES para guiones largos
# 💡 Úsala si tu guion es muy largo (+5 minutos de audio)
# ============================================================

import os
import soundfile as sf
import numpy as np
from f5_tts.api import F5TTS
from google.colab import files

# Tu guion dividido en párrafos (cada elemento = una parte del audio)
GUION_EN_PARTES = [
    """Esta es la primera parte del guion. Bienvenidos al canal.""",
    """Esta es la segunda parte. El misterio comienza aquí.""",
    """Esta es la tercera parte. El clímax de la historia.""",
]

f5tts = F5TTS()
partes_audio = []
sample_rate = 24000

for i, parte in enumerate(GUION_EN_PARTES):
    print(f'🔄 Generando parte {i+1}/{len(GUION_EN_PARTES)}...')
    output_parte = f'/content/parte_{i+1}.wav'
    
    wav, sr, _ = f5tts.infer(
        ref_file=ref_audio_path,
        ref_text=TEXTO_DEL_AUDIO_REFERENCIA,
        gen_text=parte,
        file_wave=output_parte,
        seed=42
    )
    
    data, sr = sf.read(output_parte)
    partes_audio.append(data)
    print(f'   ✅ Parte {i+1} lista.')

# Silencio de 0.5 segundos entre partes
silencio = np.zeros(int(sample_rate * 0.5))

# Unir todas las partes
audio_final = []
for i, parte in enumerate(partes_audio):
    audio_final.append(parte)
    if i < len(partes_audio) - 1:
        audio_final.append(silencio)

audio_completo = np.concatenate(audio_final)
OUTPUT_COMPLETO = '/content/guion_completo.wav'
sf.write(OUTPUT_COMPLETO, audio_completo, sample_rate)

print()
print(f'✅ ¡Guion completo generado! Duración: {len(audio_completo)/sample_rate:.1f} segundos')

import IPython.display as ipd
display(ipd.Audio(OUTPUT_COMPLETO))
files.download(OUTPUT_COMPLETO)
print('📥 Descarga iniciada: guion_completo.wav')

---
## 🆘 Solución de problemas comunes

| Error | Solución |
|---|---|
| `CUDA out of memory` | Tu guion es muy largo. Usa la **Celda Extra** para dividirlo en partes |
| `ref_audio_path not defined` | Ejecuta la **Celda 2** para subir tu audio |
| Audio suena raro o robótico | Asegúrate que el texto de referencia coincide **exactamente** con lo que dices en el audio |
| No se detectó GPU | Ve a `Entorno de ejecución` → `Cambiar tipo de entorno` → **GPU T4** |
| La celda de instalación falla | Reinicia el entorno (`Entorno de ejecución` → `Reiniciar sesión`) y vuelve desde la Celda 1 |

---
## 💡 Consejos para mejor calidad de voz

1. **Audio de referencia**: Graba 30 segundos específicamente para esto, en silencio absoluto, con buena dicción
2. **Texto de referencia**: Que coincida al 100% con lo que dices en el audio
3. **Guion**: Usa puntuación correcta (comas, puntos) — el modelo las usa para las pausas
4. **Velocidad**: Si hablas muy rápido en el audio de referencia, el modelo generará audio rápido también

---
*Notebook creado para el flujo de trabajo de creación de videos de terror 🩸*